# Opis projektu i dobór priorów

Ten notebook opisuje aktualną wersję projektu, czyli **modele tabelowe** przewidujące końcową tabelę Premier League. Starsze modele bramkowe nie są tutaj omawiane, bo nie są już używane w finalnym podejściu.

W projekcie używamy dwóch modeli tabelowych:

- `stan/table_static.stan` - model statyczny z jednym globalnym skillem drużyny,
- `stan/table_hierarchical.stan` - model hierarchiczny z trwałą jakością drużyny i sezonowym odchyleniem.

Oba modele przewidują liczbę punktów drużyny w sezonie, a następnie z symulowanych punktów budowana jest przewidywana tabela.

In [ ]:
import numpy as np
import pandas as pd

from helping_functions import (
    load_matches,
    load_season_tables,
    prepare_table_stan_static,
    prepare_table_stan_hierarchical,
    BACKTEST_TRAIN_SEASONS,
    BACKTEST_TEST_SEASON,
    FORECAST_TRAIN_SEASONS,
    FORECAST_SEASON_LABEL,
    STUDENT_T_NU,
)

matches = load_matches()
tables = load_season_tables(matches, BACKTEST_TRAIN_SEASONS)

print(f"Mecze w danych: {len(matches)}")
print(f"Sezony w danych: {sorted(matches['season'].unique())}")
print(f"Sezony treningowe backtestu: {BACKTEST_TRAIN_SEASONS[0]}-{BACKTEST_TRAIN_SEASONS[-1]}")
print(f"Sezon testowy backtestu: {BACKTEST_TEST_SEASON}")
print(f"Sezony treningowe finalnego forecastu: {FORECAST_TRAIN_SEASONS[0]}-{FORECAST_TRAIN_SEASONS[-1]}")
print(f"Forecast sezonu: {FORECAST_SEASON_LABEL}")
print(f"Student-t nu w modelach tabelowych: {STUDENT_T_NU}")

## Cel projektu

Celem jest probabilistyczne przewidywanie tabeli Premier League na kolejny sezon. Modele nie przewidują pojedynczych wyników meczów. Zamiast tego modelują końcową liczbę punktów dla każdej drużyny.

Przepływ jest następujący:

1. Wczytujemy historyczne mecze.
2. Z meczów budujemy końcowe tabele sezonów.
3. Dla każdej pary drużyna-sezon wyliczamy cechy opisujące jakość i formę.
4. Trenujemy model Stan na punktach końcowych.
5. Symulujemy punkty dla drużyn w sezonie prognozowanym.
6. Sortujemy symulowane punkty i otrzymujemy rozkład możliwych pozycji w tabeli.

Wynik nie jest jedną tabelą, tylko rozkładem: mediany pozycji, średnie punkty i niepewność predykcji.

## Dane wejściowe

Źródłem danych są pliki `season-*.csv` z katalogu `Data/datahub.io`. Każdy wiersz to jeden mecz. Do modeli tabelowych używamy tych danych po agregacji do poziomu drużyna-sezon.

Najpierw funkcja `compute_table` buduje końcową tabelę sezonu. Następnie `load_season_tables(..., with_features=True)` dokleja cechy procesowe. Domyślnie używane są **lagowane cechy**, czyli dla targetu z sezonu `s` model dostaje informacje z sezonu `s-1`.

Jedna obserwacja dla modelu tabelowego oznacza:

```text
jedna drużyna w jednym sezonie -> końcowe punkty + cechy
```

Dla 16 sezonów treningowych po 20 drużyn mamy 320 obserwacji. Lagowanie cech jest ważne, bo zapobiega przeciekowi informacji: model nie widzi strzałów celnych ani formy z sezonu, którego punkty ma przewidzieć.

In [ ]:
tables[["season", "team", "Pts", "sot_diff_pg", "pts_lag1", "ppg_last10"]].head(10)

## Cechy używane w modelach

Zmienna objaśniana:

- `Pts` - końcowa liczba punktów drużyny w sezonie.

Predyktory:

- `sot_diff_pg` - różnica strzałów celnych na mecz z poprzedniego sezonu,
- `pts_lag1` - punkty z poprzedniego sezonu; dla beniaminków i pierwszego sezonu w danych przyjmujemy 0,
- `ppg_last10` - punkty na mecz w ostatnich 10 kolejkach poprzedniego sezonu.

Wszystkie trzy predyktory są standaryzowane do z-score w funkcji `_standardize_features`. Dzięki temu współczynniki regresji mówią o zmianie oczekiwanych punktów przy zmianie cechy o jedno odchylenie standardowe. Standaryzacja jest liczona tylko na danych treningowych, a potem te same średnie i odchylenia są używane dla forecastu.

In [ ]:
tables[["Pts", "sot_diff_pg", "pts_lag1", "ppg_last10"]].describe().round(2)

## Dlaczego Student-t?

Oba modele tabelowe używają rozkładu Studenta:

$$Pts_n \sim t_{\nu}(\mu_n, \sigma_{pts})$$

W projekcie `nu = 5` jest stałe. Student-t jest bardziej odporny na nietypowe sezony niż rozkład normalny. To ma sens dla tabel ligowych, bo zdarzają się sezony odstające: bardzo mocny mistrz, wyjątkowo słaby spadkowicz, nietypowy sezon beniaminka albo drużyna z dużymi problemami kadrowymi.

`sigma_pts` nie oznacza surowego odchylenia standardowego punktów w tabeli. Oznacza skalę reszt po uwzględnieniu cech i skillu.

## Model 1: `table_static.stan`

Model statyczny zakłada, że każda drużyna ma jeden latentny skill w całym okresie treningowym.

$$Pts_{s,t} \sim t_{\nu}(\mu_{s,t}, \sigma_{pts})$$

$$\mu_{s,t} = \alpha + \beta_{pts} skill_t + \beta_{sot} x^{sot}_{s,t} + \beta_{lag} x^{lag}_{s,t} + \beta_{form} x^{form}_{s,t}$$

Do Stana wrzucamy:

- `N` - liczba obserwacji drużyna-sezon,
- `T` - liczba drużyn obecnych w treningu,
- `team[n]` - indeks drużyny,
- `pts[n]` - końcowe punkty,
- `sot_diff_pg[n]`, `pts_lag1[n]`, `ppg_last10[n]` - wystandaryzowane cechy,
- `nu` - stopnie swobody rozkładu Studenta.

Ten model jest prosty i stabilny. Dobrze działa, jeśli stała jakość drużyny jest silnym sygnałem, a różnice między sezonami da się częściowo opisać cechami.

In [ ]:
static_data, static_team_to_idx, static_teams, static_feature_stats = prepare_table_stan_static(
    tables,
    BACKTEST_TRAIN_SEASONS,
)

{k: (len(v) if hasattr(v, "__len__") and k not in {"N", "T", "nu"} else v) for k, v in static_data.items()}

### Priory w `table_static.stan`

```stan
intercept ~ normal(52, 10);
beta_pts ~ normal(20, 10);
beta_sot ~ normal(0, 8);
beta_lag ~ normal(0, 0.5);
beta_form ~ normal(0, 8);
log_sigma_pts ~ normal(log(17), 0.3);
skill ~ std_normal();
```

`intercept ~ normal(52, 10)` reprezentuje przeciętną drużynę Premier League. Średnia liczba punktów w lidze jest w okolicach środka tabeli, zwykle blisko 50 kilku punktów.

`skill ~ std_normal()` nadaje drużynom latentną jakość na skali standardowej. W Stan używamy `sum_to_zero_vector[T]`, więc średnia skillu jest równa zero i model jest identyfikowalny bez dodatkowego centrowania.

`beta_pts ~ normal(20, 10)` przeskalowuje latentny skill na punkty. Jeżeli drużyna jest o jedną jednostkę latentnego skilla wyżej, może to oznaczać kilkanaście lub kilkadziesiąt punktów różnicy.

`beta_sot ~ normal(0, 8)` i `beta_form ~ normal(0, 8)` są dość szerokie, bo cechy są standaryzowane. Zmiana o jedno odchylenie standardowe może oznaczać kilka punktów, ale prior dopuszcza też większy efekt.

`beta_lag ~ normal(0, 0.5)` jest węższy, bo `pts_lag1` częściowo dubluje informację o jakości drużyny. Szeroki prior mógłby sprawić, że poprzedni sezon zdominuje predykcję.

`log_sigma_pts ~ normal(log(17), 0.3)` startuje blisko surowego rozrzutu punktów w tabeli. Posterior może zejść niżej, jeśli skill i cechy wyjaśniają dużo zmienności. Parametr ma szerokie ograniczenie dodatnie przez log-skalę, żeby uniknąć niestabilnych wartości podczas inicjalizacji.

## Model 2: `table_hierarchical.stan`

Model hierarchiczny jest rozszerzeniem modelu tabelowego. Obecna wersja rozdziela skill na dwie części:

$$skill_{s,t} = team\_skill_t + season\_dev_{s,t}$$

- `team_skill[t]` - trwała jakość drużyny,
- `season_dev[s,t]` - małe odchylenie w konkretnym sezonie.

Równanie modelu:

$$Pts_{s,t} \sim t_{\nu}(\mu_{s,t}, \sigma_{pts})$$

$$\mu_{s,t} = \alpha + team\_skill_t + season\_dev_{s,t} + \beta_{sot} x^{sot}_{s,t} + \beta_{lag} x^{lag}_{s,t} + \beta_{form} x^{form}_{s,t}$$

To jest kompromis między modelem statycznym a modelem w pełni sezonowym. Model zachowuje informację, że drużyny mają trwałą jakość, ale pozwala też na sezonowe zmiany.

In [ ]:
hier_data, hier_team_to_idx, hier_teams, hier_season_to_idx, hier_feature_stats = prepare_table_stan_hierarchical(
    tables,
    BACKTEST_TRAIN_SEASONS,
)

{k: (len(v) if hasattr(v, "__len__") and k not in {"N", "S", "T", "K", "nu"} else v) for k, v in hier_data.items()}

### Co dodatkowo trafia do modelu hierarchicznego?

Poza danymi z modelu statycznego model hierarchiczny dostaje:

- `S` - liczba sezonów,
- `K` - liczba drużyn w jednym sezonie, zwykle 20,
- `season[n]` - indeks sezonu obserwacji,
- `obs_pos[n]` - pozycja obserwacji w obrębie sezonu.

`obs_pos` jest potrzebne, bo sezonowe odchylenia są zapisane jako `array[S] sum_to_zero_vector[K]`. Dzięki temu w każdym sezonie model ma dokładnie 20 odchyleń i ich suma wynosi zero.

To poprawia geometrię samplowania. Wcześniejszy wariant z pełną macierzą `S x T` tworzył wartości dla drużyn, które nie grały w danym sezonie. Takie puste komórki pogarszały identyfikowalność i mieszały w skali sezonowego skilla.

### Priory w `table_hierarchical.stan`

```stan
intercept ~ normal(52, 10);
beta_sot ~ normal(0, 8);
beta_lag ~ normal(0, 0.5);
beta_form ~ normal(0, 8);
log_sigma_pts ~ normal(log(4.5), 0.12);
log_tau_team ~ normal(log(7), 0.20);
log_tau_season ~ normal(log(2.25), 0.18);
team_skill_z ~ std_normal();
season_dev_z[s] ~ std_normal();
```

`log_sigma_pts ~ normal(log(4.5), 0.12)` opisuje resztkowy szum punktowy po uwzględnieniu cech, trwałej jakości drużyny i sezonowego odchylenia. Ta skala jest dużo niższa niż surowe SD punktów w tabeli, bo część zmienności wyjaśniają predyktory i skill.

`log_tau_team ~ normal(log(7), 0.20)` opisuje rozrzut trwałej jakości drużyn. Prior jest umiarkowany: pozwala modelowi pamiętać, że niektóre drużyny są systematycznie mocniejsze, ale nie pozwala temu komponentowi całkowicie zdominować cech.

`log_tau_season ~ normal(log(2.25), 0.18)` shrinkuje sezonowe odchylenia. To jest celowe, bo dla jednej drużyny w jednym sezonie mamy tylko jedną obserwację punktów. Zbyt duży sezonowy skill łatwo łapie szum zamiast sygnału.

Parametry są zapisane na log-skali, żeby skale były dodatnie. Dodatkowo mają szerokie ograniczenia domenowe, np. `log_sigma_pts` między `log(0.5)` i `log(20)`, co poprawia stabilność numeryczną inicjalizacji.

## Dlaczego model hierarchiczny nie powinien być zbyt elastyczny?

Model tabelowy ma mało obserwacji: jedna liczba punktów na drużynę w sezonie. Jeśli każdej parze sezon-drużyna damy prawie niezależny skill, model może bardzo dobrze dopasować historię, ale gorzej prognozować przyszłość.

Dlatego finalna konstrukcja jest taka:

```text
skill = trwała jakość drużyny + małe sezonowe odchylenie
```

Trwała jakość przenosi informację między sezonami. Sezonowe odchylenie pozwala na zmianę formy, ale jest regularizowane przez prior `tau_season`.

## Backtest i finalny forecast

W podstawowym backteście trenujemy modele na sezonach do 2024/25 i przewidujemy sezon 2025/26. Dzięki temu można porównać predykcję z rzeczywistą tabelą.

Szersze porównanie modeli jest w notebooku `05_rolling_backtest_table_models.ipynb`, gdzie testujemy kilka kolejnych sezonów. To jest mocniejsza walidacja niż jeden sezon testowy.

W finalnym forecastcie trenujemy na wszystkich dostępnych sezonach, włącznie z 2025/26, i przewidujemy sezon 2026/27.

Dla drużyn prognozowanego sezonu cechy są budowane na podstawie ostatniego dostępnego sezonu. Drużyny, których nie było w treningu albo nie grały w poprzednim sezonie Premier League, dostają neutralny skill i średnie wartości cech po standaryzacji. To jest uczciwe pod kątem braku przecieku, ale może być sportowo zbyt optymistyczne dla beniaminków.

## Diagnostyka modeli

Po dopasowaniu modelu sprawdzamy:

- `R_hat` - powinno być blisko 1.00, zwykle akceptujemy do 1.01,
- `ESS_bulk` i `ESS_tail` - im większe, tym lepiej; niskie ESS oznacza słabe mieszanie łańcuchów,
- divergences - powinno być 0,
- treedepth - nie powinien masowo dobijać do maksimum,
- E-BFMI - niskie wartości sugerują problemy z geometrią posterioru.

Diagnostyka samplera mówi, czy posterior został dobrze wysamplowany. Nie mówi jeszcze, czy forecast jest dobry. Jakość predykcji sprawdzamy osobno przez backtest: błędy pozycji, błędy punktów i porównanie przewidywanej tabeli z rzeczywistą.

## Krótkie porównanie modeli tabelowych

`table_static`:

- prostszy,
- stabilny diagnostycznie,
- ma jeden skill drużyny dla wszystkich sezonów,
- często daje dobrą predykcję, bo mocno korzysta z trwałej jakości drużyn.

`table_hierarchical`:

- bardziej elastyczny,
- rozdziela trwały skill drużyny i sezonowe odchylenie,
- wymaga mocniejszej regularizacji,
- może lepiej uchwycić zmiany formy, ale tylko jeśli sezonowe odchylenia nie są zbyt szerokie.

W praktyce model hierarchiczny nie powinien być oceniany po samej złożoności. Trzeba patrzeć na backtest. Bardziej złożony model może gorzej przewidywać, jeśli dodatkowa elastyczność łapie szum historyczny.

## Praca w Dockerze

Środowisko Stan działa w kontenerze Docker o nazwie `stan`. Projekt jest zamontowany w kontenerze pod ścieżką:

```text
/workspace/DA_Project
```

Przykładowa komenda:

```bash
docker exec -w /workspace/DA_Project stan python -c "import cmdstanpy; print(cmdstanpy.cmdstan_path())"
```

Dzięki temu notebooki, `cmdstanpy` i modele Stan działają w tym samym środowisku.